# Kernel Ridge Regression (RBF) in PyTorch

This notebook fits a 1D analytic function from noisy samples using **kernel ridge regression (KRR)** with an **RBF (Gaussian) kernel**.

## Model class (representer form)
Kernel methods search for predictors in the finite span
$$
\hat f(x) = \sum_{i=1}^N \alpha_i\, k(x, x_i),
$$
where $\{x_i\}_{i=1}^N$ are the training inputs and $k$ is a positive definite kernel.

## RBF kernel and its hyperparameters
We use the **RBF kernel**
$$
k(x,z) = \sigma^2\,\exp\!\left(-\frac{\|x-z\|^2}{2\,\ell^2}\right),
$$
with hyperparameters:

- $\ell$ (**lengthscale**): controls smoothness / how quickly correlation decays with distance.
  - small $\ell$ → very local influence → potentially **wiggly** fit;
  - large $\ell$ → broad influence → **smoother** fit.
- $\sigma^2$ (**kernel amplitude**): overall scale of the kernel (we fix $\sigma^2=1$ here).

## Training objective (ridge-regularized least squares)
KRR solves the regularized least-squares problem
$$
\min_{f\in\mathcal H_k}\; \sum_{i=1}^N \bigl(f(x_i)-y_i\bigr)^2 \, +\, \lambda\,\|f\|_{\mathcal H_k}^2,
$$
where $\mathcal H_k$ is the RKHS induced by $k$.

Let $K\in\mathbb{R}^{N\times N}$ be the Gram matrix $K_{ij}=k(x_i,x_j)$ and $y\in\mathbb{R}^N$ the data vector.
The coefficients satisfy the linear system
$$
(K + \lambda I)\,\alpha = y.
$$

### Interpreting $\lambda$
- $\lambda$ (**ridge / Tikhonov regularization**): trades off data-fit vs smoothness (RKHS norm).
  - small $\lambda$ → closer interpolation (can overfit noise);
  - large $\lambda$ → stronger smoothing / bias.

## Prediction
Given a test point $x_*$, define $k_*\in\mathbb{R}^N$ with entries $(k_*)_i=k(x_*,x_i)$.
Then
$$
\hat f(x_*) = k_*^\top\alpha.
$$

## What the later sweep does
Near the end we sweep over a small grid of $(\ell,\lambda)$ values to visualize how **lengthscale** and **regularization strength** affect the fit.


In [ ]:
import math
import torch
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_dtype(torch.float64)  # nicer numerics for linear solves
print("device:", device)


In [ ]:
# Analytic function on R -> R
def f_true(x: torch.Tensor) -> torch.Tensor:
    # x: (N,1)
    return torch.sin(2.0 * math.pi * x) + 0.3 * torch.cos(6.0 * math.pi * x)

# Training data
torch.manual_seed(0)
N = 40
noise_std = 0.10

x_train = torch.linspace(0.0, 1.0, N).reshape(-1, 1).to(device)
y_clean = f_true(x_train)
y_train = y_clean + noise_std * torch.randn_like(y_clean)

# Test grid for plotting
x_test = torch.linspace(-0.2, 1.2, 500).reshape(-1, 1).to(device)
y_test_true = f_true(x_test)

print("x_train:", x_train.shape, "y_train:", y_train.shape)


In [ ]:
def rbf_kernel(x1: torch.Tensor, x2: torch.Tensor, ell: float, sigma2: float = 1.0) -> torch.Tensor:
    """RBF kernel: k(x,z) = sigma2 * exp(-||x-z||^2 / (2 ell^2))
    x1: (N,d), x2: (M,d) -> (N,M)
    """
    x1_sq = (x1**2).sum(dim=1, keepdim=True)          # (N,1)
    x2_sq = (x2**2).sum(dim=1, keepdim=True).T        # (1,M)
    dist2 = x1_sq + x2_sq - 2.0 * (x1 @ x2.T)          # (N,M)
    return sigma2 * torch.exp(-0.5 * dist2 / (ell**2))

ell = 0.15       # lengthscale
lam = 1e-3       # ridge / Tikhonov regularization

K = rbf_kernel(x_train, x_train, ell=ell)          # (N,N)
k_test = rbf_kernel(x_test, x_train, ell=ell)      # (Ntest,N)

print("K:", K.shape, "k_test:", k_test.shape)


In [ ]:
# Solve (K + lam I) alpha = y
I = torch.eye(N, device=device, dtype=K.dtype)
alpha = torch.linalg.solve(K + lam * I, y_train)   # (N,1)

# Predict on test points: fhat(x_test) = k_test @ alpha
y_pred = k_test @ alpha

print("alpha:", alpha.shape, "y_pred:", y_pred.shape)


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(x_test.detach().cpu(), y_test_true.detach().cpu(), label="true")
plt.scatter(x_train.detach().cpu(), y_train.detach().cpu(), s=30, label="train (noisy)")
plt.plot(x_test.detach().cpu(), y_pred.detach().cpu(), label="kernel ridge (RBF)")
plt.legend()
plt.title(f"Kernel ridge regression (ell={ell}, lam={lam})")
plt.xlabel("x")
plt.ylabel("y")
plt.grid(True)
plt.show()


## Optional: quick sweep over $(\ell,\lambda)$

Recall:
- $\ell$ is the **RBF lengthscale** (smaller → more local/wiggly; larger → smoother).
- $\lambda$ is the **ridge regularization** (smaller → closer fit; larger → more smoothing).

The grid below is not "optimal" tuning—it's just to build intuition.


In [ ]:
ells = [0.05, 0.15, 0.40]
lams = [1e-6, 1e-3, 1e-1]

plt.figure(figsize=(10, 6))
plot_id = 1
for ell_ in ells:
    for lam_ in lams:
        K_ = rbf_kernel(x_train, x_train, ell=ell_)
        alpha_ = torch.linalg.solve(K_ + lam_ * torch.eye(N, device=device, dtype=K_.dtype), y_train)
        y_pred_ = rbf_kernel(x_test, x_train, ell=ell_) @ alpha_

        plt.subplot(len(ells), len(lams), plot_id)
        plt.plot(x_test.detach().cpu(), y_test_true.detach().cpu(), linewidth=1)
        plt.scatter(x_train.detach().cpu(), y_train.detach().cpu(), s=10)
        plt.plot(x_test.detach().cpu(), y_pred_.detach().cpu(), linewidth=2)
        plt.title(f"ell={ell_}, lam={lam_}")
        plt.xticks([])
        plt.yticks([])
        plot_id += 1

plt.tight_layout()
plt.show()
